# Build a Child-Like AI Agent in Python

This notebook combines the workshop explanation and the runnable prototype in one place.

Core idea:
- The agent learns by updating its own identity.
- There is no separate long-term memory system.
- There is no reward-based reinforcement learning.

Learning loop:
`perceive -> decide -> feedback -> evaluate usefulness -> update identity`


## Overview

In this codelab-style notebook, you will build a minimal prototype of a child-like AI agent that learns continuously from interaction.

Unlike many AI systems, this prototype does **not** use:
- large pretrained models
- reward-based reinforcement learning
- a separate long-term memory database

Instead, the agent evolves by updating its internal identity state over time.


## Why This Is Not Reinforcement Learning

In reinforcement learning, an agent usually:
- takes an action
- receives a scalar reward
- updates a policy or value estimate to maximize future reward

This notebook does something different.

Here, the agent:
- perceives an input
- makes a prediction with confidence
- receives feedback
- judges whether the feedback is useful
- updates its identity directly

There is no reward signal, no policy gradient, and no optimization loop.


## Step 1: Imports and Data Structures

We start with a small set of standard-library imports and a `Decision` dataclass.


In [1]:
from __future__ import annotations

from copy import deepcopy
from dataclasses import dataclass
from pprint import pformat
from typing import Any, Dict, List, Optional


LINE = "=" * 78
DIVIDER = "-" * 78


@dataclass
class Decision:
    """A simple prediction returned by the agent."""

    prediction: str
    confidence: float
    reason: str


## Step 2: Identity Store

The identity store is the heart of the system.

It includes:
- `beliefs`: learned categories and examples
- `interaction_count`: total interactions seen
- `learning_count`: how many times identity changed
- `ignored_feedback_count`: how many times feedback was skipped
- `confidence_threshold`: when to say "I don't know"
- `last_update_reason`: a human-readable explanation of the last change

This matters because the agent does not store a separate memory log. Its learning is represented directly in identity.


In [2]:
class ChildLikeAgent:
    """
    A tiny identity-based learning agent.

    The identity store is the central state of the system.
    Instead of writing to a separate memory database, the agent updates its
    internal beliefs directly as part of who it is.
    """

    def __init__(self) -> None:
        self.identity: Dict[str, Any] = {
            "name": "Milo",
            "version": "codelab-prototype-1",
            "beliefs": {},
            "interaction_count": 0,
            "learning_count": 0,
            "ignored_feedback_count": 0,
            "confidence_threshold": 0.60,
            "last_decision": None,
            "last_update_reason": "Agent created.",
        }

    def perceive(self, text: str) -> Dict[str, Any]:
        """
        Turn raw input into a tiny internal representation.

        This keeps the prototype lightweight and easy to explain live.
        """
        normalized = text.strip().lower()
        tokens = [token.strip(".,!?") for token in normalized.split() if token.strip(".,!?")]

        return {
            "raw_text": text,
            "normalized_text": normalized,
            "tokens": tokens,
            "token_count": len(tokens),
        }

    def decide(self, perception: Dict[str, Any]) -> Decision:
        """
        Predict the best matching category from current identity.

        Confidence is based on token overlap with previously learned examples
        plus a small boost from how often that category has been reinforced.
        """
        beliefs = self.identity["beliefs"]
        input_tokens = set(perception["tokens"])

        if not beliefs:
            decision = Decision(
                prediction="I don't know yet.",
                confidence=0.0,
                reason="The agent has no learned beliefs yet.",
            )
            self.identity["last_decision"] = decision.prediction
            return decision

        best_category: Optional[str] = None
        best_score = 0.0
        best_reason = "No useful overlap with known examples."

        for category, belief in beliefs.items():
            example_tokens = set()
            for example in belief["examples"]:
                example_tokens.update(example.split())

            overlap = len(input_tokens & example_tokens)
            coverage = overlap / max(len(input_tokens), 1)
            strength_bonus = min(1.0, 0.4 + 0.1 * belief["strength"])
            score = coverage * strength_bonus

            if score > best_score:
                best_category = category
                best_score = score
                best_reason = (
                    f"Matched {overlap} token(s) with '{category}' examples; "
                    f"belief strength={belief['strength']}."
                )

        threshold = self.identity["confidence_threshold"]
        if best_category is None or best_score < threshold:
            decision = Decision(
                prediction="I don't know.",
                confidence=round(best_score, 2),
                reason=best_reason,
            )
            self.identity["last_decision"] = decision.prediction
            return decision

        decision = Decision(
            prediction=best_category,
            confidence=round(best_score, 2),
            reason=best_reason,
        )
        self.identity["last_decision"] = decision.prediction
        return decision

    def evaluate_usefulness(
        self,
        perception: Dict[str, Any],
        decision: Decision,
        feedback_label: str,
    ) -> Dict[str, Any]:
        """
        Decide whether the feedback should change the identity.

        This is intentionally not reward-based RL.
        The agent is not maximizing a scalar reward.
        It is judging whether the incoming feedback is novel or corrective.
        """
        beliefs = self.identity["beliefs"]
        normalized_text = perception["normalized_text"]
        known_belief = beliefs.get(feedback_label)

        is_new_category = known_belief is None
        is_new_example = not known_belief or normalized_text not in known_belief["examples"]
        was_uncertain = decision.confidence < self.identity["confidence_threshold"]
        was_wrong = decision.prediction not in {
            feedback_label,
            "I don't know.",
            "I don't know yet.",
        }

        useful = is_new_category or is_new_example or was_uncertain or was_wrong

        reasons: List[str] = []
        if is_new_category:
            reasons.append("new category")
        if is_new_example:
            reasons.append("new example")
        if was_uncertain:
            reasons.append("agent was uncertain")
        if was_wrong:
            reasons.append("agent was wrong")
        if not reasons:
            reasons.append("redundant feedback")

        return {"useful": useful, "reason": ", ".join(reasons)}

    def learn(
        self,
        perception: Dict[str, Any],
        feedback_label: str,
        usefulness: Dict[str, Any],
    ) -> bool:
        """
        Update identity directly when the feedback is judged useful.
        """
        self.identity["interaction_count"] += 1

        if not usefulness["useful"]:
            self.identity["ignored_feedback_count"] += 1
            self.identity["last_update_reason"] = f"No learning: {usefulness['reason']}."
            return False

        beliefs = self.identity["beliefs"]
        example = perception["normalized_text"]

        if feedback_label not in beliefs:
            beliefs[feedback_label] = {"examples": [], "strength": 0}

        if example not in beliefs[feedback_label]["examples"]:
            beliefs[feedback_label]["examples"].append(example)

        beliefs[feedback_label]["strength"] += 1
        self.identity["learning_count"] += 1
        self.identity["last_update_reason"] = (
            f"Learned '{feedback_label}' because {usefulness['reason']}."
        )
        return True


## Step 3: Demo Helpers and Simulation Data

The next cell adds a small dataset and helper functions so the workshop output is easy to follow.


In [3]:
def get_demo_interactions() -> List[Dict[str, str]]:
    """Small workshop dataset that shows uncertainty, learning, and redundancy."""
    return [
        {"input": "apple is a fruit", "feedback": "fruit"},
        {"input": "banana is a fruit", "feedback": "fruit"},
        {"input": "carrot is a vegetable", "feedback": "vegetable"},
        {"input": "spinach is a vegetable", "feedback": "vegetable"},
        {"input": "dog is an animal", "feedback": "animal"},
        {"input": "pear is a fruit", "feedback": "fruit"},
        {"input": "truck is a vehicle", "feedback": "vehicle"},
        {"input": "cat is an animal", "feedback": "animal"},
        {"input": "lettuce is a vegetable", "feedback": "vegetable"},
        {"input": "plane is a vehicle", "feedback": "vehicle"},
        {"input": "mango is a fruit", "feedback": "fruit"},
        {"input": "rock is a mineral", "feedback": "mineral"},
        {"input": "banana is a fruit", "feedback": "fruit"},
    ]


def print_banner() -> None:
    print(LINE)
    print("CHILD-LIKE AI AGENT: GOOGLE CODELAB DEMO")
    print(LINE)
    print("This prototype learns by updating identity from interaction.")
    print("It is not reinforcement learning: no reward, no policy, no value function.")
    print()


def print_identity_diff(before: Dict[str, Any], after: Dict[str, Any]) -> None:
    print("\nIDENTITY CHANGES")
    changed = False

    for key in after:
        if before.get(key) != after.get(key):
            changed = True
            print(f"- {key}:")
            print(f"  before: {pformat(before.get(key))}")
            print(f"  after : {pformat(after.get(key))}")

    if not changed:
        print("- No identity changes")


def print_interaction_summary(
    index: int,
    perception: Dict[str, Any],
    decision: Decision,
    feedback: str,
    usefulness: Dict[str, Any],
    learned: bool,
    identity: Dict[str, Any],
) -> None:
    print(DIVIDER)
    print(f"INTERACTION {index}")
    print(f"Perception: {perception['raw_text']}")
    print(f"Tokens    : {perception['tokens']}")
    print(f"Decision  : {decision.prediction}")
    print(f"Confidence: {decision.confidence:.2f}")
    print(f"Reason    : {decision.reason}")
    print(f"Feedback  : {feedback}")
    print(f"Useful?   : {usefulness['useful']}")
    print(f"Why learn : {usefulness['reason']}")
    print(f"Learned?  : {learned}")
    print(f"Update    : {identity['last_update_reason']}")


def print_final_summary(agent: ChildLikeAgent) -> None:
    print(LINE)
    print("FINAL SUMMARY")
    print(LINE)
    print(f"Interactions seen : {agent.identity['interaction_count']}")
    print(f"Learning updates  : {agent.identity['learning_count']}")
    print(f"Ignored feedback  : {agent.identity['ignored_feedback_count']}")
    print("Known categories  :", ", ".join(agent.identity["beliefs"].keys()))


def run_demo() -> ChildLikeAgent:
    agent = ChildLikeAgent()
    print_banner()

    for index, item in enumerate(get_demo_interactions(), start=1):
        before_identity = deepcopy(agent.identity)

        perception = agent.perceive(item["input"])
        decision = agent.decide(perception)
        usefulness = agent.evaluate_usefulness(perception, decision, item["feedback"])
        learned = agent.learn(perception, item["feedback"], usefulness)

        print_interaction_summary(
            index=index,
            perception=perception,
            decision=decision,
            feedback=item["feedback"],
            usefulness=usefulness,
            learned=learned,
            identity=agent.identity,
        )
        print_identity_diff(before_identity, agent.identity)
        print("\nCURRENT IDENTITY SNAPSHOT")
        print(pformat(agent.identity, sort_dicts=False))
        print()

    print_final_summary(agent)
    return agent


## Step 4: Run the Full Learning Loop

This cell runs the complete simulation.

Watch for:
- early low-confidence `I don't know` behavior
- identity changes after feedback
- later higher-confidence predictions
- redundant feedback being ignored


In [4]:
agent = run_demo()

CHILD-LIKE AI AGENT: GOOGLE CODELAB DEMO
This prototype learns by updating identity from interaction.
It is not reinforcement learning: no reward, no policy, no value function.

------------------------------------------------------------------------------
INTERACTION 1
Perception: apple is a fruit
Tokens    : ['apple', 'is', 'a', 'fruit']
Decision  : I don't know yet.
Confidence: 0.00
Reason    : The agent has no learned beliefs yet.
Feedback  : fruit
Useful?   : True
Why learn : new category, new example, agent was uncertain
Learned?  : True
Update    : Learned 'fruit' because new category, new example, agent was uncertain.

IDENTITY CHANGES
- beliefs:
  before: {}
  after : {'fruit': {'examples': ['apple is a fruit'], 'strength': 1}}
- interaction_count:
  before: 0
  after : 1
- learning_count:
  before: 0
  after : 1
- last_decision:
  before: None
  after : "I don't know yet."
- last_update_reason:
  before: 'Agent created.'
  after : "Learned 'fruit' because new category, new ex

## Step 5: Inspect the Final Identity

The cell below prints the final identity store so you can inspect what the agent has become after the interactions.


In [5]:
print(pformat(agent.identity, sort_dicts=False))

{'name': 'Milo',
 'version': 'codelab-prototype-1',
 'beliefs': {'fruit': {'examples': ['apple is a fruit',
                                    'banana is a fruit',
                                    'pear is a fruit',
                                    'mango is a fruit'],
                       'strength': 4},
             'vegetable': {'examples': ['carrot is a vegetable',
                                        'spinach is a vegetable',
                                        'lettuce is a vegetable'],
                           'strength': 3},
             'animal': {'examples': ['dog is an animal', 'cat is an animal'],
                        'strength': 2},
             'vehicle': {'examples': ['truck is a vehicle',
                                      'plane is a vehicle'],
                         'strength': 2},
             'mineral': {'examples': ['rock is a mineral'], 'strength': 1}},
 'interaction_count': 13,
 'learning_count': 12,
 'ignored_feedback_count': 1,
 'confi

## Workshop Checkpoint

After running the notebook, make sure you can explain these four points:

1. Why the agent says `I don't know` early on.
2. Why repeated feedback can later be ignored as redundant.
3. Why updating identity is different from storing a separate memory log.
4. Why this prototype is not reinforcement learning.
